# LOF Anomaly Detection

This notebook walks through preprocessing, feature engineering, Local Outlier Factor (LOF) training, evaluation, and visualization for wind turbine gearbox data.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import PathConfig, SENSOR_FEATURE_COLUMNS, TIMESTAMP_COLUMN

np.random.seed(42)
plt.style.use("seaborn-v0_8")

In [ ]:
data_path = PathConfig.RAW_DATA / "turbine_5yr_complex_data.csv"

raw_df = pd.read_csv(data_path, parse_dates=[TIMESTAMP_COLUMN])
raw_df = raw_df.sort_values(TIMESTAMP_COLUMN).drop_duplicates(subset=TIMESTAMP_COLUMN)
raw_df = raw_df.reset_index(drop=True)

raw_df.head()

In [ ]:
missing_summary = raw_df[SENSOR_FEATURE_COLUMNS].isna().mean().sort_values(ascending=False)
missing_summary

In [ ]:
clean_df = raw_df.dropna(subset=SENSOR_FEATURE_COLUMNS).copy()

rolling_window = 60
for column in SENSOR_FEATURE_COLUMNS:
    clean_df[f"{column}_roll_mean"] = clean_df[column].rolling(rolling_window).mean()
    clean_df[f"{column}_roll_std"] = clean_df[column].rolling(rolling_window).std()

clean_df = clean_df.dropna().reset_index(drop=True)

engineered_features = [
    *SENSOR_FEATURE_COLUMNS,
    *[f"{column}_roll_mean" for column in SENSOR_FEATURE_COLUMNS],
    *[f"{column}_roll_std" for column in SENSOR_FEATURE_COLUMNS],
]

clean_df[engineered_features].head()

In [ ]:
train_frac = 0.7
val_frac = 0.15

n_samples = len(clean_df)
train_end = int(n_samples * train_frac)
val_end = int(n_samples * (train_frac + val_frac))

train_df = clean_df.iloc[:train_end]
val_df = clean_df.iloc[train_end:val_end]
test_df = clean_df.iloc[val_end:]

train_df.shape, val_df.shape, test_df.shape